# Precio contra avalúo catastral

In [1]:
from eda_utils import *

El avalúo de la base es una sola foto del catastro: el 96% de los predios vendidos en
años distintos tiene el mismo avalúo en todas sus ventas. Por eso llevamos el precio a
pesos de 2022 (el año de esa foto) antes de dividirlo por el avalúo. Una razón de 1
significa que el catastro coincide con el mercado. El 68% de las viviendas tiene avalúo
(23.696 de 34.606): el 52% de ellas son del IGAC y el 40% del observatorio.

In [2]:
# comprobar que el avalúo es de un solo año
misma_foto = (df.dropna(subset=["avaluo", "codigo_predial"]).groupby("codigo_predial")
                .agg(anios=("anio", "nunique"), avaluos=("avaluo", "nunique")).query("anios > 1"))
print(f"Predios vendidos en años distintos: {len(misma_foto):,}; con el mismo avalúo en todas sus ventas: "
      f"{(misma_foto.avaluos == 1).mean():.0%}")

a_2022 = ipc_anio.loc[2022] / ipc_anio
brecha = viv[viv.avaluo.notna()].copy()
brecha["razon"] = brecha.precio * brecha.anio.map(a_2022) / brecha.avaluo
# Con 6 o más veces el avalúo, el avalúo de 2022 no describe la vivienda vendida: en la mitad de
# esos predios el catastro de 2022 tenía 0 m² construidos (era el lote, antes de construir).
desactualizado = brecha.razon >= 6
print(f"Excluidas por avalúo desactualizado (precio de 6 o más veces el avalúo): {desactualizado.sum():,} "
      f"({desactualizado.mean():.1%})")
brecha = brecha[brecha.razon.between(0.1, 6, inclusive="left")]
r = brecha.razon

fig = make_subplots(1, 2, subplot_titles=("Distribución de la razón precio / avalúo",
                                          "Razón mediana por estrato"), horizontal_spacing=0.12)
fig.add_histogram(x=r, nbinsx=60, marker_color=PRINCIPAL, row=1, col=1, showlegend=False,
                  hovertemplate="%{x:.1f} veces: %{y} ventas<extra></extra>")
fig.add_vline(x=1, line_dash="dash", line_color="black", row=1, col=1, annotation_text="precio = avalúo")

# razón mediana por estrato, con todas las ventas
mediana_estrato = brecha.groupby("estrato", observed=True).razon.median().reindex(ESTRATOS)
fig.add_bar(x=[n.replace("Estrato ", "") for n in NOMBRES_ESTRATO], y=mediana_estrato,
            marker_color=COLORES_ESTRATO, showlegend=False,
            text=mediana_estrato.round(2), textposition="outside", cliponaxis=False,
            hovertemplate="estrato %{x}: %{y:.2f} veces el avalúo<extra></extra>", row=1, col=2)
fig.add_hline(y=1, line_dash="dash", line_color="black", row=1, col=2,
              annotation_text="precio = avalúo", annotation_position="bottom right")
fig.update_xaxes(title_text="veces el avalúo (precio en pesos de 2022)", row=1, col=1)
fig.update_yaxes(title_text="número de ventas", row=1, col=1)
fig.update_xaxes(title_text="estrato", row=1, col=2)
fig.update_yaxes(title_text="veces el avalúo (mediana)", range=[0, 2], row=1, col=2)
fig.update_layout(bargap=0.35)
mostrar(fig, "¿Cuánto se paga por encima del avalúo catastral?", alto=480)

print(f"Ventas con avalúo: {len(brecha):,}   mediana: {r.median():.2f}   "
      f"por encima del avalúo: {(r > 1).mean():.0%}   al doble o más: {(r >= 2).mean():.0%}")
W, p = stats.wilcoxon(np.log(r), alternative="greater")
anotar("¿Se paga por encima del avalúo?", "Wilcoxon de rangos con signo", W, p, f"mediana = {r.median():.2f}")
H, p, eps2 = kruskal_por_grupo(brecha, "razon", "estrato", ESTRATOS)
anotar("¿La razón precio/avalúo cambia con el estrato?", "Kruskal-Wallis", H, p, f"épsilon² = {eps2:.2f}")
d = brecha.dropna(subset=["estrato_num"])
rho, p = stats.spearmanr(d.estrato_num, d.razon)
anotar("¿A mayor estrato, mayor subvaloración?", "Correlación de Spearman", rho, p, f"rho = {rho:.2f}")
brecha.groupby("estrato_txt").razon.median().round(2).rename("razón mediana").to_frame().T

Predios vendidos en años distintos: 1,425; con el mismo avalúo en todas sus ventas: 96%
Excluidas por avalúo desactualizado (precio de 6 o más veces el avalúo): 284 (1.2%)


Ventas con avalúo: 21,462   mediana: 1.44   por encima del avalúo: 79%   al doble o más: 19%
Wilcoxon de rangos con signo: estadístico = 189,736,076.000   p = < 0,001   mediana = 1.44
Kruskal-Wallis: estadístico = 260.585   p = < 0,001   épsilon² = 0.01
Correlación de Spearman: estadístico = -0.015   p = 0,027   rho = -0.02


estrato_txt,Estrato 1,Estrato 2,Estrato 3,Estrato 4,Estrato 5,Estrato 6,Sin estrato
razón mediana,1.48,1.26,1.44,1.56,1.26,1.30,1.09


In [3]:
por_loc = (brecha[brecha.localidad.notna() & ~brecha.localidad.isin(NO_BARRIOS)]
           .groupby("localidad").razon.median().sort_values())
por_bar = (brecha[brecha.barrio.notna() & ~brecha.barrio.isin(NO_BARRIOS)]
           .groupby("barrio").razon.agg(["median", "size"]).query("size >= 30").sort_values("median"))

fig = make_subplots(1, 2, subplot_titles=("Por localidad", "Barrios con 30 ventas o más (extremos)"),
                    horizontal_spacing=0.25)
fig.add_bar(x=por_loc.values, y=por_loc.index.str.title(), orientation="h", marker_color=PRINCIPAL, row=1, col=1,
            hovertemplate="%{y}: %{x:.2f} veces<extra></extra>")
extremos = pd.concat([por_bar.head(6), por_bar.tail(6)])
fig.add_bar(x=extremos["median"], y=extremos.index.str.title(), orientation="h", row=1, col=2,
            marker_color=[PRINCIPAL] * 6 + [OSCURO] * 6, hovertemplate="%{y}: %{x:.2f} veces<extra></extra>")
for c in (1, 2):
    fig.add_vline(x=1, line_dash="dash", line_color="black", row=1, col=c)
fig.update_xaxes(title_text="razón mediana")
fig.update_layout(showlegend=False)
mostrar(fig, "La brecha con el avalúo según la zona", alto=460)
print(f"Localidades: de {por_loc.min():.2f} ({por_loc.index[0].title()}) a {por_loc.max():.2f} ({por_loc.index[-1].title()})")
print(f"Barrios: de {por_bar['median'].min():.2f} ({por_bar.index[0].title()}) a "
      f"{por_bar['median'].max():.2f} ({por_bar.index[-1].title()})")

Localidades: de 1.19 (Suroriente) a 1.51 (Suroccidente)
Barrios: de 0.93 (Santodomingo) a 2.12 (Zona De Expansion Riomar)


Dejamos por fuera 284 ventas (1,2%) cuyo precio es 6 o más veces el avalúo. En ellas el avalúo
no describe la vivienda vendida: en la mitad de los casos el catastro de 2022 tenía el predio con
0 m² construidos, es decir, valoraba el lote antes de que se construyera.

La vivienda típica se vende a 1,44 veces su avalúo, el 79% por encima de él y una de cada cinco
al doble o más. Por estrato, la razón mediana va de 1,26 a 1,56, sin un orden claro.

La razón cambia mucho según la fuente: es 1,14 en el IGAC, frente a 1,71 en el observatorio y
1,81 en las georreferenciadas. En el IGAC muchas escrituras registran un precio muy cercano al
avalúo; ese es un problema aparte, que no se corrige aquí.

El estrato explica muy poco de la brecha (épsilon² = 0,01). Entre zonas va de 1,19 en Suroriente
a 1,51 en Suroccidente, y entre barrios de 0,9 a 2,1 (el más alto es la Zona de Expansión Riomar,
de obra nueva), aunque estas cifras también están afectadas por cuántas ventas del IGAC tiene
cada zona.